In [1]:
import os
import warnings
import logging

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/three-model-utils/')
sys.path.append('/kaggle/input/cmi-competition-code')

import os
import json
import importlib
from datetime import datetime

import numpy as np
import pandas as pd

import data_utils
import utils_hierarchical_aug as utils
import utils as base_utils

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real

E0000 00:00:1778798631.838930      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778798631.906121      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778798632.439516      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778798632.439556      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778798632.439559      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778798632.439562      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = data_utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
# ============================================================
# Experiment config
# ============================================================

n_splits = 3
cv = GroupKFold(n_splits=n_splits)

pipe_name = 'temporal_extractor'
corrector_name = 'orientation_corrector'
classifier_name = 'multi_branch'

# only these two modes in this notebook
search_mode = "grid"  # grid or bayesian
candidates = 10

holdout_size = 0.20
random_state = 42

# Keep the three hierarchy targets explicit
target_flag_col = "is_target"
orientation_target_col = "orientation"
action_target_col = "gesture_action"
final_eval_target_col = "hierarchical_gesture"

experiment_notes = "three_model_hierarchy_target_orientation_action_with_augmentation_params"

handedness_lookup = train_demo_df.set_index("subject")["handedness"].to_dict()

In [4]:
# ============================================================
# Prepare dataframe and hierarchy labels
# No manual sensor-correction code here. The pipeline corrector handles that.
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df["is_target"] = train_df["sequence_type"].eq("Target").astype(int)

# Three-model hierarchy evaluates target gestures exactly, but collapses all non-target sequences.
# If you need full non-target gesture labels too, add a fourth non-target gesture model.
train_df["hierarchical_gesture"] = np.where(
    train_df["sequence_type"].eq("Target"),
    train_df["gesture"],
    "Non-Target",
)

# Mapping used to reconstruct final target gesture from orientation + gesture_action
orientation_action_map_df = (
    train_df
    .loc[train_df["sequence_type"].eq("Target")]
    .drop_duplicates(["orientation", "gesture_action", "gesture"])
    [["orientation", "gesture_action", "gesture"]]
    .copy()
)

mapping_check = (
    orientation_action_map_df
    .groupby(["orientation", "gesture_action"])["gesture"]
    .nunique()
    .reset_index(name="n_gesture_labels")
)

print("Mapping uniqueness check")
print(mapping_check["n_gesture_labels"].value_counts(dropna=False))
print(mapping_check.sort_values("n_gesture_labels", ascending=False).head())

orientation_action_lookup = {
    (row.orientation, row.gesture_action): row.gesture
    for row in orientation_action_map_df.itertuples(index=False)
}

fallback_target_gesture = (
    train_df
    .loc[train_df["sequence_type"].eq("Target"), "gesture"]
    .mode()
    .iloc[0]
)

print("rows:", len(train_df))
print("sequences:", train_df["sequence_id"].nunique())
print("subjects:", train_df["subject"].nunique())
print(train_df[["sequence_type", "hierarchical_gesture"]].drop_duplicates()["hierarchical_gesture"].value_counts().head())

Mapping uniqueness check
n_gesture_labels
2    8
3    4
1    4
Name: count, dtype: int64
                        orientation gesture_action  n_gesture_labels
1                       Lie on Back      pull hair                 3
5        Lie on Side - Non Dominant      pull hair                 3
13                  Seated Straight      pull hair                 3
9   Seated Lean Non Dom - FACE DOWN      pull hair                 3
7        Lie on Side - Non Dominant        scratch                 2
rows: 574945
sequences: 8151
subjects: 81
hierarchical_gesture
Cheek - pinch skin          1
Forehead - pull hairline    1
Non-Target                  1
Neck - scratch              1
Neck - pinch skin           1
Name: count, dtype: int64


In [5]:
# ============================================================
# Subject-grouped holdout split
# ============================================================

seq_meta = (
    train_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture"]]
    .reset_index(drop=True)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=holdout_size,
    random_state=random_state,
)

train_seq_idx, holdout_seq_idx = next(
    splitter.split(
        seq_meta,
        y=seq_meta["hierarchical_gesture"],
        groups=seq_meta["subject"],
    )
)

train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

train_model_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
holdout_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

target_only_train_df = train_model_df.loc[train_model_df["sequence_type"].eq("Target")].copy()

target_only_holdout_df = holdout_df.loc[holdout_df["sequence_type"].eq("Target")].copy()

print("train sequences:", train_model_df["sequence_id"].nunique())
print("holdout sequences:", holdout_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())
print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
print("holdout subjects:", holdout_df["subject"].nunique())

train sequences: 6422
holdout sequences: 1729
target-only train sequences: 4025
target-only holdout sequences: 1088
holdout subjects: 17


In [6]:
# ============================================================
# Search spaces: GRID or BAYESIAN only
# These include augmentation params on the classifier.
# ============================================================

if search_mode == "bayesian":
    param_space = {
        # --- Preprocessing ---
        f"{pipe_name}__acc_mode": Categorical(["raw", "jerk", "velocity"]),
        f"{pipe_name}__linear_acc_mode": Categorical([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([False, True]),
        f"{pipe_name}__sampling_rate": Categorical([20, 25, 50]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([None, 20.0, 50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__window_size": Integer(3, 21),
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.5]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__include_mask": Categorical([False]),
        f"{pipe_name}__rotation_mode": Categorical(["quaternion", "angular_velocity", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([True]),
        f"{pipe_name}__tof_mode": Categorical([None, "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_255"]),
        f"{pipe_name}__thm_mode": Categorical([None, "centered_diff"]),

        # --- Classifier ---
        f"{classifier_name}__maxlen": Integer(16, 160),
        f"{classifier_name}__padding_value": Categorical([-999.0]),
        f"{classifier_name}__fusion_mode": Categorical(["attention", "bigru", "none"]),
        f"{classifier_name}__attention_heads": Integer(2, 8),
        f"{classifier_name}__gru_units": Integer(64, 256),
        f"{classifier_name}__branch_filters": [
            {"acc":"64", "rot":"32", "tof":"16", "thm":"8"},
            {"acc":"64-128", "rot":"32-64", "tof":"32", "thm":"16"},
            {"acc":"128", "rot":"64", "tof":"32", "thm":"16"},
        ],
        f"{classifier_name}__branch_kernel_sizes": [
            {"acc":"3", "rot":"3", "tof":"3", "thm":"3"},
            {"acc":"3-3", "rot":"3-3", "tof":"3", "thm":"3"},
        ],
        f"{classifier_name}__branch_pool_sizes": [
            {"acc":"none", "rot":"none", "tof":"none", "thm":"none"},
            {"acc":"none-none", "rot":"none-none", "tof":"none", "thm":"none"},
        ],
        f"{classifier_name}__use_batch_norm": Categorical([True, False]),
        f"{classifier_name}__spatial_dropout": Real(0.0, 0.3),
        f"{classifier_name}__dense_units": Categorical(["32", "64", "128", "64-32"]),
        f"{classifier_name}__dropout": Real(0.0, 0.5),
        f"{classifier_name}__learning_rate": Real(1e-4, 2e-3, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32, 64]),
        f"{classifier_name}__epochs": Categorical([60, 80]),
        f"{classifier_name}__patience": Categorical([8, 12]),

        # --- Augmentation params ---
        f"{classifier_name}__use_mixup": Categorical([False, True]),
        f"{classifier_name}__mixup_alpha": Real(0.2, 0.6),
        f"{classifier_name}__mixup_size": Real(0.5, 1.0),
        f"{classifier_name}__use_time_shift": Categorical([False, True]),
        f"{classifier_name}__time_shift_max_pct": Real(0.05, 0.25),
        f"{classifier_name}__use_time_stretch": Categorical([False, True]),
        f"{classifier_name}__time_stretch_min_rate": Real(0.5, 0.9),
        f"{classifier_name}__time_stretch_max_rate": Real(1.1, 1.5),
        f"{classifier_name}__use_noise": Categorical([False, True]),
        f"{classifier_name}__noise_std": Real(0.001, 0.03, prior="log-uniform"),
        f"{classifier_name}__use_magnitude_scaling": Categorical([False, True]),
        f"{classifier_name}__use_time_mask": Categorical([False, True]),
        f"{classifier_name}__time_mask_ratio": Real(0.05, 0.15),
        f"{classifier_name}__use_tof_dropout": Categorical([False, True]),
        f"{classifier_name}__tof_dropout_prob": Real(0.05, 0.2),
        f"{classifier_name}__use_channel_dropout": Categorical([False, True]),
        f"{classifier_name}__channel_dropout_prob": Real(0.02, 0.1),
        f"{classifier_name}__use_modality_dropout": Categorical([False, True]),
        f"{classifier_name}__modality_dropout_prob": Real(0.05, 0.2),
        f"{classifier_name}__use_quat_sign_flip": Categorical([False, True]),
        f"{classifier_name}__use_cutmix": Categorical([False, True]),
        f"{classifier_name}__cutmix_prob": Real(0.1, 0.4),
        f"{classifier_name}__cutmix_ratio": Real(0.15, 0.4),
    }

    param_space = base_utils.prepare_multibranch_param_space(
        param_space,
        search_mode,
        Categorical=Categorical,
    )

elif search_mode == "grid":
    param_space = {
        # --- Preprocessing ---
        f"{pipe_name}__acc_mode": ["raw"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__sampling_rate": [20],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": [5],
        f"{pipe_name}__smooth_alpha": [None],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__include_mask": [False],
        f"{pipe_name}__rotation_mode": ["rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [True],
        f"{pipe_name}__tof_mode": ["sensor_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate"],
        f"{pipe_name}__thm_mode": ["centered_diff"],

        # --- Classifier ---
        f"{classifier_name}__maxlen": [120],
        f"{classifier_name}__padding_value": [-999.0],
        f"{classifier_name}__fusion_mode": ["attention"],
        f"{classifier_name}__attention_heads": [4],
        f"{classifier_name}__gru_units": [128],
        f"{classifier_name}__branch_filters": [
            {"acc":"64", "rot":"32", "tof":"16", "thm":"8"},
        ],
        f"{classifier_name}__branch_kernel_sizes": [
            {"acc":"3", "rot":"3", "tof":"3", "thm":"3"},
        ],
        f"{classifier_name}__branch_pool_sizes": [
            {"acc":"none", "rot":"none", "tof":"none", "thm":"none"},
        ],
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.1],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.3],
        f"{classifier_name}__learning_rate": [5e-4],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [80],
        f"{classifier_name}__patience": [12],

        # --- Augmentation params ---
        f"{classifier_name}__use_mixup": [False, True],
        f"{classifier_name}__mixup_alpha": [0.4],
        f"{classifier_name}__mixup_size": [1.0],
        f"{classifier_name}__use_time_shift": [False],
        f"{classifier_name}__time_shift_max_pct": [0.25],
        f"{classifier_name}__use_time_stretch": [False],
        f"{classifier_name}__time_stretch_min_rate": [0.5],
        f"{classifier_name}__time_stretch_max_rate": [1.5],
        f"{classifier_name}__use_noise": [False],
        f"{classifier_name}__noise_std": [0.01],
        f"{classifier_name}__use_magnitude_scaling": [False],
        f"{classifier_name}__magnitude_scale_min": [0.9],
        f"{classifier_name}__magnitude_scale_max": [1.1],
        f"{classifier_name}__use_time_mask": [False],
        f"{classifier_name}__time_mask_ratio": [0.1],
        f"{classifier_name}__use_tof_dropout": [False],
        f"{classifier_name}__tof_dropout_prob": [0.1],
        f"{classifier_name}__use_channel_dropout": [False],
        f"{classifier_name}__channel_dropout_prob": [0.05],
        f"{classifier_name}__use_modality_dropout": [False],
        f"{classifier_name}__modality_dropout_prob": [0.1],
        f"{classifier_name}__use_quat_sign_flip": [False],
        f"{classifier_name}__use_cutmix": [False],
        f"{classifier_name}__cutmix_prob": [0.25],
        f"{classifier_name}__cutmix_ratio": [0.3],
        f"{classifier_name}__use_frequency_filter": [False],
        f"{classifier_name}__freq_keep_min": [0.1],
        f"{classifier_name}__freq_keep_max": [0.9],
    }

else:
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

In [7]:
# ============================================================
# Base pipeline shape
# ============================================================

importlib.reload(utils)

base_pipeline = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(
        handedness_lookup=handedness_lookup,
        correct_handedness=True,
        correct_upside_down=True,
    )),
    (pipe_name, utils.SequenceExtractor()),
    (classifier_name, utils.KerasAugmentedMultiBranchClassifier(
        target=target_flag_col,
        fusion_mode="attention",
        batch_size=32,
        epochs=80,
        patience=12,
        random_state=random_state,
    )),
])

In [8]:
# ============================================================
# Model 1: Target vs Non-Target
# ============================================================

target_pipeline = clone(base_pipeline)
target_pipeline.set_params(**{f"{classifier_name}__target": target_flag_col})

if search_mode == "bayesian":
    target_search = BayesSearchCV(
        estimator=target_pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=random_state,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )
else:
    target_search = GridSearchCV(
        estimator=target_pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )

y_target = train_model_df[["sequence_id", target_flag_col]].copy()
groups_target = train_model_df["subject"]

target_search.fit(train_model_df, y_target, groups=groups_target)

print("Target detector best score:", target_search.best_score_)
print("Target detector best params:")
print(target_search.best_params_)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


I0000 00:00:1778799313.760335      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778799313.766532      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778799319.895020      56 service.cc:152] XLA service 0x7d53e400b710 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778799319.895073      56 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778799319.895080      56 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778799320.936309      56 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778799327.738553      56 device_compiler.h:188] Compiled clust

[CV 1/3] END multi_branch__attention_heads=4, multi_branch__batch_size=32, multi_branch__branch_filters={'acc': '64', 'rot': '32', 'tof': '16', 'thm': '8'}, multi_branch__branch_kernel_sizes={'acc': '3', 'rot': '3', 'tof': '3', 'thm': '3'}, multi_branch__branch_pool_sizes={'acc': 'none', 'rot': 'none', 'tof': 'none', 'thm': 'none'}, multi_branch__channel_dropout_prob=0.05, multi_branch__cutmix_prob=0.25, multi_branch__cutmix_ratio=0.3, multi_branch__dense_units=64, multi_branch__dropout=0.3, multi_branch__epochs=80, multi_branch__freq_keep_max=0.9, multi_branch__freq_keep_min=0.1, multi_branch__fusion_mode=attention, multi_branch__gru_units=128, multi_branch__learning_rate=0.0005, multi_branch__magnitude_scale_max=1.1, multi_branch__magnitude_scale_min=0.9, multi_branch__maxlen=120, multi_branch__mixup_alpha=0.4, multi_branch__mixup_size=1.0, multi_branch__modality_dropout_prob=0.1, multi_branch__noise_std=0.01, multi_branch__padding_value=-999.0, multi_branch__patience=12, multi_branc

In [9]:
# ============================================================
# Model 2: Orientation, trained on target-only sequences
# ============================================================

orientation_pipeline = clone(base_pipeline)
orientation_pipeline.set_params(**{f"{classifier_name}__target": orientation_target_col})

if search_mode == "bayesian":
    orientation_search = BayesSearchCV(
        estimator=orientation_pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=random_state,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )
else:
    orientation_search = GridSearchCV(
        estimator=orientation_pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )

y_orientation = target_only_train_df[["sequence_id", orientation_target_col]].copy()
groups_orientation = target_only_train_df["subject"]

orientation_search.fit(target_only_train_df, y_orientation, groups=groups_orientation)

print("Orientation model best score:", orientation_search.best_score_)
print("Orientation model best params:")
print(orientation_search.best_params_)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV 1/3] END multi_branch__attention_heads=4, multi_branch__batch_size=32, multi_branch__branch_filters={'acc': '64', 'rot': '32', 'tof': '16', 'thm': '8'}, multi_branch__branch_kernel_sizes={'acc': '3', 'rot': '3', 'tof': '3', 'thm': '3'}, multi_branch__branch_pool_sizes={'acc': 'none', 'rot': 'none', 'tof': 'none', 'thm': 'none'}, multi_branch__channel_dropout_prob=0.05, multi_branch__cutmix_prob=0.25, multi_branch__cutmix_ratio=0.3, multi_branch__dense_units=64, multi_branch__dropout=0.3, multi_branch__epochs=80, multi_branch__freq_keep_max=0.9, multi_branch__freq_keep_min=0.1, multi_branch__fusion_mode=attention, multi_branch__gru_units=128, multi_branch__learning_rate=0.0005, multi_branch__magnitude_scale_max=1.1, multi_branch__magnitude_scale_min=0.9, multi_branch__maxlen=120, multi_branch__mixup_alpha=0.4, multi_branch__mixup_size=1.0, multi_branch__modality_dropout_prob=0.1, multi_branch__noise_std=0.01, multi_branch__p

In [10]:
# ============================================================
# Model 3: Gesture action, trained on target-only sequences
# ============================================================

action_pipeline = clone(base_pipeline)
action_pipeline.set_params(**{f"{classifier_name}__target": action_target_col})

if search_mode == "bayesian":
    action_search = BayesSearchCV(
        estimator=action_pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=random_state,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )
else:
    action_search = GridSearchCV(
        estimator=action_pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv,
        n_jobs=1,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan,
    )

y_action = target_only_train_df[["sequence_id", action_target_col]].copy()
groups_action = target_only_train_df["subject"]

action_search.fit(target_only_train_df, y_action, groups=groups_action)

print("Gesture action model best score:", action_search.best_score_)
print("Gesture action model best params:")
print(action_search.best_params_)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV 1/3] END multi_branch__attention_heads=4, multi_branch__batch_size=32, multi_branch__branch_filters={'acc': '64', 'rot': '32', 'tof': '16', 'thm': '8'}, multi_branch__branch_kernel_sizes={'acc': '3', 'rot': '3', 'tof': '3', 'thm': '3'}, multi_branch__branch_pool_sizes={'acc': 'none', 'rot': 'none', 'tof': 'none', 'thm': 'none'}, multi_branch__channel_dropout_prob=0.05, multi_branch__cutmix_prob=0.25, multi_branch__cutmix_ratio=0.3, multi_branch__dense_units=64, multi_branch__dropout=0.3, multi_branch__epochs=80, multi_branch__freq_keep_max=0.9, multi_branch__freq_keep_min=0.1, multi_branch__fusion_mode=attention, multi_branch__gru_units=128, multi_branch__learning_rate=0.0005, multi_branch__magnitude_scale_max=1.1, multi_branch__magnitude_scale_min=0.9, multi_branch__maxlen=120, multi_branch__mixup_alpha=0.4, multi_branch__mixup_size=1.0, multi_branch__modality_dropout_prob=0.1, multi_branch__noise_std=0.01, multi_branch__p

In [11]:
# ============================================================
# Save CV results for each model
# Timestamp is date + hour + minute at the end of the filename
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

cv_target_df = pd.DataFrame(target_search.cv_results_)
cv_target_df["model_stage"] = "target_vs_non_target"
cv_target_df["target"] = target_flag_col
cv_target_df["search_mode"] = search_mode
cv_target_df["experiment_notes"] = experiment_notes
cv_target_path = os.path.join(model_run_folder_name, f"cv_results_target_vs_non_target_{timestamp}.csv")
cv_target_df.to_csv(cv_target_path, index=False)

cv_orientation_df = pd.DataFrame(orientation_search.cv_results_)
cv_orientation_df["model_stage"] = "orientation"
cv_orientation_df["target"] = orientation_target_col
cv_orientation_df["search_mode"] = search_mode
cv_orientation_df["experiment_notes"] = experiment_notes
cv_orientation_path = os.path.join(model_run_folder_name, f"cv_results_orientation_{timestamp}.csv")
cv_orientation_df.to_csv(cv_orientation_path, index=False)

cv_action_df = pd.DataFrame(action_search.cv_results_)
cv_action_df["model_stage"] = "gesture_action"
cv_action_df["target"] = action_target_col
cv_action_df["search_mode"] = search_mode
cv_action_df["experiment_notes"] = experiment_notes
cv_action_path = os.path.join(model_run_folder_name, f"cv_results_gesture_action_{timestamp}.csv")
cv_action_df.to_csv(cv_action_path, index=False)

print(cv_target_path)
print(cv_orientation_path)
print(cv_action_path)

model_runs/cv_results_target_vs_non_target_20260515_0342.csv
model_runs/cv_results_orientation_20260515_0342.csv
model_runs/cv_results_gesture_action_20260515_0342.csv


In [12]:
# ============================================================
# Holdout evaluation of the full three-model hierarchy
# ============================================================

best_target_model = target_search.best_estimator_
best_orientation_model = orientation_search.best_estimator_
best_action_model = action_search.best_estimator_

holdout_seq = (
    holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "sequence_type", "gesture", "hierarchical_gesture"]]
    .reset_index(drop=True)
)

is_target_pred = best_target_model.predict(holdout_df)
orientation_pred = best_orientation_model.predict(holdout_df)
action_pred = best_action_model.predict(holdout_df)

hier_pred = []

for is_t, orient, action in zip(is_target_pred, orientation_pred, action_pred):
    if int(is_t) == 1:
        hier_pred.append(
            orientation_action_lookup.get(
                (str(orient), str(action)),
                fallback_target_gesture,
            )
        )
    else:
        hier_pred.append("Non-Target")

holdout_seq["prediction"] = hier_pred

hier_f1 = f1_score(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["prediction"],
    average="macro",
)

print("--- Final Hierarchical Holdout Results ---")
print("Target detector CV:", target_search.best_score_)
print("Orientation CV:", orientation_search.best_score_)
print("Gesture action CV:", action_search.best_score_)
print("Hierarchical holdout macro F1:", round(hier_f1, 4))
print("\nClassification report:")
print(classification_report(holdout_seq["hierarchical_gesture"], holdout_seq["prediction"]))

holdout_results_path = os.path.join(model_run_folder_name, f"holdout_results_three_model_hierarchy_{timestamp}.csv")
holdout_seq.to_csv(holdout_results_path, index=False)
print(holdout_results_path)

--- Final Hierarchical Holdout Results ---
Target detector CV: 0.9312320339683056
Orientation CV: 0.9417038566301715
Gesture action CV: 0.5830829099088197
Hierarchical holdout macro F1: 0.3516

Classification report:
                          precision    recall  f1-score   support

   Above ear - pull hair       0.20      0.18      0.19       136
      Cheek - pinch skin       0.32      0.30      0.31       136
     Eyebrow - pull hair       0.20      0.35      0.25       136
     Eyelash - pull hair       0.17      0.17      0.17       136
Forehead - pull hairline       0.65      0.46      0.54       136
      Forehead - scratch       0.44      0.15      0.22       136
       Neck - pinch skin       0.34      0.28      0.31       136
          Neck - scratch       0.19      0.26      0.22       136
              Non-Target       0.94      0.97      0.95       641

                accuracy                           0.53      1729
               macro avg       0.38      0.35      0.35

In [13]:
# ============================================================
# Compact summary row
# ============================================================

summary_df = pd.DataFrame([{
    "timestamp": timestamp,
    "search_mode": search_mode,
    "experiment_notes": experiment_notes,
    "target_detector_cv": target_search.best_score_,
    "orientation_cv": orientation_search.best_score_,
    "gesture_action_cv": action_search.best_score_,
    "hierarchical_holdout_macro_f1": hier_f1,
    "n_train_sequences": train_model_df["sequence_id"].nunique(),
    "n_holdout_sequences": holdout_df["sequence_id"].nunique(),
    "n_target_train_sequences": target_only_train_df["sequence_id"].nunique(),
    "n_target_holdout_sequences": target_only_holdout_df["sequence_id"].nunique(),
}])

summary_path = os.path.join(model_run_folder_name, f"summary_three_model_hierarchy_{timestamp}.csv")
summary_df.to_csv(summary_path, index=False)
summary_df

,timestamp,search_mode,experiment_notes,target_detector_cv,orientation_cv,gesture_action_cv,hierarchical_holdout_macro_f1,n_train_sequences,n_holdout_sequences,n_target_train_sequences,n_target_holdout_sequences
0,20260515_0342,grid,three_model_hierarchy_target_orientation_actio...,0.931232,0.941704,0.583083,0.351621,6422,1729,4025,1088
